# DUC & DUS Family — Benchmarks

Benchmarks D-U-S, D-U-S-D, D-U-S-M, D-U-C, D-U-C-T, D-U-C-M-r, and D-U-C-M-b together, using the same fixed sample for all algorithms within a benchmark (seed 42). Ray is initialised once before any timing so startup cost is excluded.  
Each benchmark writes to **two** CSVs — DUS-family rows go to `experiments/experiment_results_dus/`, DUC-family rows go to `experiments/experiment_results_duc/` — both overwritten from scratch on every run.

In [1]:
import sys, os
sys.path.insert(0, '../src')

import csv
import functools
import time
from datetime import datetime

import numpy as np
import ray

from generator_multidim import MultidimSampleGenerator
from dus import discover_dus
from dusd import discover_dus_dimension
from dusm import discover_dusm
from duc import discover_duc
from duct import discover_duc_tree
from ducm import discover_ducm, partition_traces_naive, partition_traces_by_length

DUS_RESULTS_DIR = '../experiments/experiment_results_dus'
DUC_RESULTS_DIR = '../experiments/experiment_results_duc'

DUS_ALGORITHMS = [
    ('D-U-S',   discover_dus),
    ('D-U-S-D', discover_dus_dimension),
    ('D-U-S-M', discover_dusm),
]
DUC_ALGORITHMS = [
    ('D-U-C',     discover_duc),
    ('D-U-C-T',   discover_duc_tree),
    ('D-U-C-M-r', functools.partial(discover_ducm, partition_fn=partition_traces_naive)),
    ('D-U-C-M-b', functools.partial(discover_ducm, partition_fn=partition_traces_by_length)),
]

# (name, fn, family) — family selects which of the two output CSVs a row goes to.
ALGORITHMS = (
    [(name, fn, 'dus') for name, fn in DUS_ALGORITHMS] +
    [(name, fn, 'duc') for name, fn in DUC_ALGORITHMS]
)

/Users/Rosie/.pyenv/versions/3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-13 00:07:56,713	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Initialise Ray BEFORE timing so startup overhead is excluded from measurements.
if not ray.is_initialized():
    ray.init(runtime_env={"env_vars": {"PYTHONPATH": os.path.abspath('../src')}})
print("Ray ready:", ray.is_initialized())

2026-07-13 00:08:11,817	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8266 


Ray ready: True


/Users/Rosie/.pyenv/versions/3.10.12/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


(_per_domain_duc pid=1607) | Current query: $x0;; $x0;; $x0;;; stack size: 3; query count: 3
(pid=gcs_server) [2026-07-13 14:11:53,471 E 1261 44975326] (gcs_server) gcs_actor_scheduler.cc:499: Failed to kill actor 6aecf6c82404a75a2619baf401000000, return status: Invalid: KillActor RPC failed for actor 6aecf6c82404a75a2619baf401000000: RpcError: RPC error: Socket closed rpc_code: 14
(pid=gcs_server) [2026-07-13 14:11:53,702 E 1261 44975326] (gcs_server) gcs_actor_scheduler.cc:499: Failed to kill actor 5d0173e84095734710ae2c7601000000, return status: Invalid: KillActor RPC failed for actor 5d0173e84095734710ae2c7601000000: RpcError: RPC error: Socket closed rpc_code: 14
(pid=gcs_server) [2026-07-13 14:11:55,081 E 1261 44975326] (gcs_server) gcs_actor_scheduler.cc:499: Failed to kill actor 5cc8ab50c3b6e02116eb61fd01000000, return status: Invalid: KillActor RPC failed for actor 5cc8ab50c3b6e02116eb61fd01000000: RpcError: RPC error: Socket closed rpc_code: 14
(pid=gcs_server) [2026-07-13 14

In [3]:
def run_benchmark(param_name, param_values, fixed_params, algorithms, reps, seed,
                   dus_csv_path, duc_csv_path):
    """Generic benchmark loop.

    param_name:   the axis being varied ('trace_count' or 'trace_length')
    param_values: list of values to iterate over
    fixed_params: dict with the non-varying generator arguments
    algorithms:   list of (name, fn, family) with family in {'dus', 'duc'}

    Overwrites both csv paths from scratch; each algorithm's rows go to the
    csv matching its family.
    """
    gen = MultidimSampleGenerator()
    run_timestamp = datetime.now().isoformat()

    fieldnames = [
        'run_timestamp', 'algorithm', 'trace_count', 'trace_length',
        'type_count', 'dimensions', 'support', 'rep', 'time_s', 'queries_found',
    ]
    csv_paths = {'dus': dus_csv_path, 'duc': duc_csv_path}

    files = {family: open(path, 'w', newline='') for family, path in csv_paths.items()}
    writers = {family: csv.DictWriter(f, fieldnames=fieldnames) for family, f in files.items()}
    for writer in writers.values():
        writer.writeheader()

    try:
        for value in param_values:
            params = {**fixed_params, param_name: value}

            # same sample for all algorithms
            np.random.seed(seed)
            sample = gen.generate_random_sample(
                sample_size=params['trace_count'],
                min_trace_length=params['trace_length'],
                max_trace_length=params['trace_length'],
                event_dimension=params['dimensions'],
                type_count=params['type_count'],
            )

            for alg_name, alg_fn, family in algorithms:
                times = []
                queries_found = None

                for rep in range(1, reps + 1):
                    t0 = time.perf_counter()
                    result = alg_fn(sample=sample, supp=params['support'], max_query_length=-1)
                    elapsed = time.perf_counter() - t0
                    times.append(elapsed)
                    queries_found = len(result.get('queryset', set()))

                    writers[family].writerow({
                        'run_timestamp': run_timestamp,
                        'algorithm':     alg_name,
                        'trace_count':   params['trace_count'],
                        'trace_length':  params['trace_length'],
                        'type_count':    params['type_count'],
                        'dimensions':    params['dimensions'],
                        'support':       params['support'],
                        'rep':           rep,
                        'time_s':        round(elapsed, 6),
                        'queries_found': queries_found,
                    })
                    files[family].flush()

                print(f"{param_name}={value:>5}  {alg_name:<10}  "
                      f"mean={np.mean(times):.4f}s  min={min(times):.4f}s  queries={queries_found}")
    finally:
        for f in files.values():
            f.close()

    print(f"\nSaved to {dus_csv_path} and {duc_csv_path}")
    return run_timestamp

---
## Benchmark 1: varying trace count
Fixed: trace length = 10, type count = 5, dimensions = 2, supp = 1.0  
Varying: 2 000 → 50 000 (step 2 000)

In [4]:
ts_count = run_benchmark(
    param_name='trace_count',
    param_values=list(range(2000, 50001, 2000)),
    fixed_params=dict(trace_length=10, type_count=5, dimensions=2, support=1.0),
    algorithms=ALGORITHMS,
    reps=3,
    seed=42,
    dus_csv_path=os.path.join(DUS_RESULTS_DIR, 'dus_trace_count.csv'),
    duc_csv_path=os.path.join(DUC_RESULTS_DIR, 'duc_trace_count.csv'),
)

trace_count= 2000  D-U-S       mean=0.5506s  min=0.4434s  queries=2
trace_count= 2000  D-U-S-D     mean=0.6912s  min=0.3773s  queries=2
trace_count= 2000  D-U-S-M     mean=2.7892s  min=2.5119s  queries=2
trace_count= 2000  D-U-C       mean=0.4688s  min=0.4007s  queries=2
trace_count= 2000  D-U-C-T     mean=0.3478s  min=0.3259s  queries=2
trace_count= 2000  D-U-C-M-r   mean=1.3035s  min=1.2069s  queries=2
trace_count= 2000  D-U-C-M-b   mean=1.1476s  min=1.0677s  queries=2
trace_count= 4000  D-U-S       mean=1.5328s  min=1.3539s  queries=2
trace_count= 4000  D-U-S-D     mean=1.1689s  min=1.1027s  queries=2
trace_count= 4000  D-U-S-M     mean=3.1686s  min=2.8472s  queries=2
trace_count= 4000  D-U-C       mean=1.2793s  min=1.2541s  queries=2
trace_count= 4000  D-U-C-T     mean=0.9336s  min=0.9095s  queries=2
trace_count= 4000  D-U-C-M-r   mean=1.3206s  min=1.2872s  queries=2
trace_count= 4000  D-U-C-M-b   mean=1.4304s  min=1.2537s  queries=2
trace_count= 6000  D-U-S       mean=2.8392s  min

| Current query: ;$x0; ;$x0; ;$x0;; stack size: 9; query count: 3


trace_count=10000  D-U-C       mean=7.0693s  min=6.5701s  queries=2
trace_count=10000  D-U-C-T     mean=4.1067s  min=4.0810s  queries=2
trace_count=10000  D-U-C-M-r   mean=1.9752s  min=1.6962s  queries=2
trace_count=10000  D-U-C-M-b   mean=2.4783s  min=2.2291s  queries=2
trace_count=12000  D-U-S       mean=9.2841s  min=9.0149s  queries=2
trace_count=12000  D-U-S-D     mean=7.2023s  min=7.0878s  queries=2
trace_count=12000  D-U-S-M     mean=4.7259s  min=4.1135s  queries=2
trace_count=12000  D-U-C       mean=9.3875s  min=9.3635s  queries=2
trace_count=12000  D-U-C-T     mean=6.7648s  min=5.7091s  queries=2


| Current query: ;$x0; ;$x1; ;$x1; ;$x0;; stack size: 6; query count: 6


trace_count=12000  D-U-C-M-r   mean=3.0581s  min=2.8473s  queries=2
trace_count=12000  D-U-C-M-b   mean=1.9304s  min=1.8747s  queries=2
trace_count=14000  D-U-S       mean=13.0296s  min=12.6060s  queries=2
trace_count=14000  D-U-S-D     mean=10.1222s  min=10.0140s  queries=2


...

trace_count=24000  D-U-C-M-r   mean=6.0426s  min=5.5628s  queries=2
trace_count=24000  D-U-C-M-b   mean=4.5668s  min=4.1511s  queries=2
trace_count=26000  D-U-S       mean=42.1970s  min=41.6043s  queries=2
trace_count=26000  D-U-S-D     mean=33.8640s  min=32.9212s  queries=2
trace_count=26000  D-U-S-M     mean=8.4792s  min=8.3482s  queries=2


| Current query: $x0;$x1; ;$x1; $x0;;; stack size: 6; query count: 18
| Current query: $x0;; ;$x1; ;$x1; $x0;;; stack size: 3; query count: 21
| Current query: $x0;; $x1;; $x1;; $x0;;; stack size: 0; query count: 24


trace_count=26000  D-U-C       mean=42.6738s  min=42.6249s  queries=2
trace_count=26000  D-U-C-T     mean=27.4355s  min=24.9337s  queries=2
trace_count=26000  D-U-C-M-r   mean=6.0762s  min=4.9270s  queries=2
trace_count=26000  D-U-C-M-b   mean=5.8107s  min=4.6122s  queries=2


| Current query: ;$x0; ;$x1; ;$x0; ;$x1;; stack size: 1; query count: 5


trace_count=28000  D-U-S       mean=51.6571s  min=49.8675s  queries=2
trace_count=28000  D-U-S-D     mean=40.4565s  min=39.6959s  queries=2
trace_count=28000  D-U-S-M     mean=10.6821s  min=10.2022s  queries=2
trace_count=28000  D-U-C       mean=51.2735s  min=51.0838s  queries=2
trace_count=28000  D-U-C-T     mean=32.6655s  min=32.3848s  queries=2
trace_count=28000  D-U-C-M-r   mean=7.5485s  min=7.4220s  queries=2
trace_count=28000  D-U-C-M-b   mean=8.6099s  min=8.2140s  queries=2
trace_count=30000  D-U-S       mean=60.6302s  min=59.6231s  queries=2
trace_count=30000  D-U-S-D     mean=52.2315s  min=51.5454s  queries=2
trace_count=30000  D-U-S-M     mean=16.2559s  min=15.6666s  queries=2
trace_count=30000  D-U-C       mean=63.6191s  min=63.1319s  queries=2
trace_count=30000  D-U-C-T     mean=49.7014s  min=47.2850s  queries=2
trace_count=30000  D-U-C-M-r   mean=12.1286s  min=11.8116s  queries=2
trace_count=30000  D-U-C-M-b   mean=12.3393s  min=11.8053s  queries=2
trace_count=32000  D-U-S

---
## Benchmark 2: varying trace length
Fixed: trace count = 10 000, type count = 5, dimensions = 2, supp = 1.0  
Varying: trace length 1 → 12

In [5]:
ts_length = run_benchmark(
    param_name='trace_length',
    param_values=list(range(1, 13)),
    fixed_params=dict(trace_count=10000, type_count=5, dimensions=2, support=1.0),
    algorithms=ALGORITHMS,
    reps=3,
    seed=42,
    dus_csv_path=os.path.join(DUS_RESULTS_DIR, 'benchmark_dus_trace_length.csv'),
    duc_csv_path=os.path.join(DUC_RESULTS_DIR, 'benchmark_duc_trace_length.csv'),
)

trace_length=    1  D-U-S       mean=0.0249s  min=0.0113s  queries=0
trace_length=    1  D-U-S-D     mean=0.0614s  min=0.0610s  queries=0
trace_length=    1  D-U-S-M     mean=0.7721s  min=0.4942s  queries=0
trace_length=    1  D-U-C       mean=0.0158s  min=0.0155s  queries=0
trace_length=    1  D-U-C-T     mean=0.0213s  min=0.0181s  queries=0
trace_length=    1  D-U-C-M-r   mean=0.3186s  min=0.2305s  queries=0
trace_length=    1  D-U-C-M-b   mean=0.3549s  min=0.2312s  queries=0
trace_length=    2  D-U-S       mean=0.7934s  min=0.6007s  queries=0
trace_length=    2  D-U-S-D     mean=0.4397s  min=0.4296s  queries=0
trace_length=    2  D-U-S-M     mean=1.9103s  min=1.5658s  queries=0
trace_length=    2  D-U-C       mean=1.0408s  min=0.8662s  queries=0
trace_length=    2  D-U-C-T     mean=0.5806s  min=0.5086s  queries=0
trace_length=    2  D-U-C-M-r   mean=1.2282s  min=1.1368s  queries=0
trace_length=    2  D-U-C-M-b   mean=1.3034s  min=1.2038s  queries=0
trace_length=    3  D-U-S       me

---
## Benchmark 3: varying dimensions
Fixed: trace count = 10 000, trace length = 10, type count = 5, supp = 1.0  
Varying: dimensions 1 → 5

In [10]:
ts_dims = run_benchmark(
    param_name='dimensions',
    param_values=list(range(1, 6)),
    fixed_params=dict(trace_count=10000, trace_length=10, type_count=5, support=1.0),
    algorithms=ALGORITHMS,
    reps=3,
    seed=42,
    dus_csv_path=os.path.join(DUS_RESULTS_DIR, 'benchmark_dus_dimensions.csv'),
    duc_csv_path=os.path.join(DUC_RESULTS_DIR, 'benchmark_duc_dimensions.csv'),
)


#run_timestamp,algorithm,trace_count,trace_length,type_count,dimensions,support,rep,time_s,queries_found
# 2026-07-13T18:12:43.477963,D-U-S,10000,10,5,5,1.0,1,5158.208964,5
# 2026-07-13T18:12:43.477963,D-U-S,10000,10,5,5,1.0,2,7116.880056,5
# 2026-07-13T18:12:43.477963,D-U-S,10000,10,5,5,1.0,3,6462.808348,5

dimensions=    1  D-U-S       mean=1.9564s  min=1.9177s  queries=1
dimensions=    1  D-U-S-D     mean=1.9632s  min=1.9410s  queries=1
dimensions=    1  D-U-S-M     mean=2.3623s  min=1.8053s  queries=1
dimensions=    1  D-U-C       mean=1.9070s  min=1.7860s  queries=1
dimensions=    1  D-U-C-T     mean=2.1023s  min=1.9530s  queries=1
dimensions=    1  D-U-C-M-r   mean=3.3266s  min=2.5073s  queries=1
dimensions=    1  D-U-C-M-b   mean=2.4926s  min=2.2355s  queries=1
dimensions=    2  D-U-S       mean=8.9967s  min=8.3449s  queries=2
dimensions=    2  D-U-S-D     mean=7.9249s  min=6.6989s  queries=2
dimensions=    2  D-U-S-M     mean=6.2867s  min=6.1928s  queries=2
dimensions=    2  D-U-C       mean=7.5545s  min=7.4132s  queries=2
dimensions=    2  D-U-C-T     mean=5.9804s  min=5.8169s  queries=2
dimensions=    2  D-U-C-M-r   mean=3.0792s  min=2.6052s  queries=2
dimensions=    2  D-U-C-M-b   mean=3.0000s  min=2.6149s  queries=2
dimensions=    3  D-U-S       mean=20.0795s  min=18.6242s  que

---
## Benchmark 4: varying type count
Fixed: trace count = 10 000, trace length = 10, dimensions = 2, supp = 1.0  
Varying: type count 3 → 12

In [7]:
ts_types = run_benchmark(
    param_name='type_count',
    param_values=list(range(3, 13)),
    fixed_params=dict(trace_count=10000, trace_length=10, dimensions=2, support=1.0),
    algorithms=ALGORITHMS,
    reps=3,
    seed=42,
    dus_csv_path=os.path.join(DUS_RESULTS_DIR, 'benchmark_dus_type_count.csv'),
    duc_csv_path=os.path.join(DUC_RESULTS_DIR, 'benchmark_duc_type_count.csv'),
)

type_count=    3  D-U-S       mean=447.3082s  min=436.2253s  queries=10
type_count=    3  D-U-S-D     mean=441.7684s  min=423.3054s  queries=10
type_count=    3  D-U-S-M     mean=104.4062s  min=96.8840s  queries=10
type_count=    3  D-U-C       mean=133.0147s  min=128.8267s  queries=10
type_count=    3  D-U-C-T     mean=99.3864s  min=95.8161s  queries=10
type_count=    3  D-U-C-M-r   mean=32.2192s  min=31.0990s  queries=10
type_count=    3  D-U-C-M-b   mean=32.8590s  min=30.4497s  queries=10
type_count=    4  D-U-S       mean=84.8389s  min=81.9865s  queries=8
type_count=    4  D-U-S-D     mean=80.9974s  min=77.6819s  queries=8
type_count=    4  D-U-S-M     mean=23.9015s  min=22.4906s  queries=8
type_count=    4  D-U-C       mean=43.6479s  min=43.1623s  queries=8
type_count=    4  D-U-C-T     mean=30.7300s  min=28.8503s  queries=8
type_count=    4  D-U-C-M-r   mean=10.3902s  min=9.8369s  queries=8
type_count=    4  D-U-C-M-b   mean=10.8334s  min=10.0763s  queries=8
type_count=    5  D-U

---
## Benchmark 5: D-U-C-M-r vs D-U-C-M-b — widening trace-length spread
Fixed: trace count = 10 000, type count = 5, dimensions = 2, supp = 1.0  
`min_trace_length` fixed at 2; `max_trace_length` varies 2 → 100 in 10 steps, so samples go from uniform length (2) to a wide spread (2–100). This is where the naive contiguous split (D-U-C-M-r) and the length-balanced split (D-U-C-M-b) should diverge most.  
DUC-only, so written to `experiments/experiment_results_duc/`. CSV only, no plotting here.

In [12]:
DUCM_ALGORITHMS = [
    ('D-U-C-M-r', functools.partial(discover_ducm, partition_fn=partition_traces_naive)),
    ('D-U-C-M-b', functools.partial(discover_ducm, partition_fn=partition_traces_by_length)),
]


def run_benchmark_trace_length_range(min_trace_length, max_trace_length_values, fixed_params,
                                      algorithms, reps, seed, csv_path):
    """Benchmark loop where min_trace_length is fixed and max_trace_length varies,
    producing traces of non-uniform length within each sample.

    Overwrites csv_path from scratch on every call.
    """
    gen = MultidimSampleGenerator()
    run_timestamp = datetime.now().isoformat()

    fieldnames = [
        'run_timestamp', 'algorithm', 'trace_count', 'min_trace_length', 'max_trace_length',
        'type_count', 'dimensions', 'support', 'rep', 'time_s', 'queries_found',
    ]

    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for max_trace_length in max_trace_length_values:
            np.random.seed(seed)
            sample = gen.generate_random_sample(
                sample_size=fixed_params['trace_count'],
                min_trace_length=min_trace_length,
                max_trace_length=max_trace_length,
                event_dimension=fixed_params['dimensions'],
                type_count=fixed_params['type_count'],
            )

            for alg_name, alg_fn in algorithms:
                times = []
                queries_found = None

                for rep in range(1, reps + 1):
                    t0 = time.perf_counter()
                    result = alg_fn(sample=sample, supp=fixed_params['support'], max_query_length=-1)
                    elapsed = time.perf_counter() - t0
                    times.append(elapsed)
                    queries_found = len(result.get('queryset', set()))

                    writer.writerow({
                        'run_timestamp':    run_timestamp,
                        'algorithm':        alg_name,
                        'trace_count':      fixed_params['trace_count'],
                        'min_trace_length': min_trace_length,
                        'max_trace_length': max_trace_length,
                        'type_count':       fixed_params['type_count'],
                        'dimensions':       fixed_params['dimensions'],
                        'support':          fixed_params['support'],
                        'rep':              rep,
                        'time_s':           round(elapsed, 6),
                        'queries_found':    queries_found,
                    })
                    f.flush()

                print(f"max_trace_length={max_trace_length:>5}  {alg_name:<10}  "
                      f"mean={np.mean(times):.4f}s  min={min(times):.4f}s  queries={queries_found}")

    print(f"\nSaved to {csv_path}")
    return run_timestamp


run_benchmark_trace_length_range(
    min_trace_length=2,
    max_trace_length_values=[2, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],
    fixed_params=dict(trace_count=10000, type_count=5, dimensions=2, support=1.0),
    algorithms=DUCM_ALGORITHMS,
    reps=3,
    seed=42,
    csv_path=os.path.join(DUC_RESULTS_DIR, 'benchmark_ducm_max_trace_length.csv'),
)

max_trace_length=    2  D-U-C-M-r   mean=0.8917s  min=0.7675s  queries=0
max_trace_length=    2  D-U-C-M-b   mean=1.1434s  min=0.8112s  queries=0
max_trace_length=  100  D-U-C-M-r   mean=3.2152s  min=2.7843s  queries=0
max_trace_length=  100  D-U-C-M-b   mean=3.0091s  min=2.0337s  queries=0
max_trace_length=  200  D-U-C-M-r   mean=5.1232s  min=4.4340s  queries=0
max_trace_length=  200  D-U-C-M-b   mean=5.4629s  min=4.5914s  queries=0
max_trace_length=  300  D-U-C-M-r   mean=7.8870s  min=7.3034s  queries=0
max_trace_length=  300  D-U-C-M-b   mean=8.5169s  min=7.5430s  queries=0
max_trace_length=  400  D-U-C-M-r   mean=11.3108s  min=10.5019s  queries=0
max_trace_length=  400  D-U-C-M-b   mean=11.4069s  min=11.2945s  queries=0
max_trace_length=  500  D-U-C-M-r   mean=22.2176s  min=18.0506s  queries=0
max_trace_length=  500  D-U-C-M-b   mean=17.1911s  min=16.9428s  queries=0
max_trace_length=  600  D-U-C-M-r   mean=25.7983s  min=24.6921s  queries=0
max_trace_length=  600  D-U-C-M-b   mean=

'2026-07-14T10:31:28.425549'